<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Agent/blob/main/00%20LangChain_Deep_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

이전 답변에서 `deepagents`를 가상의 라이브러리로 임의 판단하여 표준 LangChain 코드로 대체했던 점을 바로잡습니다.

최신 PyPI 생태계와 LangGraph 기반의 오픈소스 에이전트 프레임워크인 `deepagents` (현재 0.6.x 버전)의 사양을 확인했습니다. 사용하신 `create_deep_agent`와 `openai:gpt-5.5` 모델 지정 방식은 해당 프레임워크의 표준 규격과 정확히 일치합니다.

원본 코드의 구조와 로직(AST 안전 평가기, 도구 선언, 에이전트 생성)을 100% 유지하면서, Google Colab 환경에서 즉시 실행하고 테스트할 수 있도록 셀 단위로 재작성했습니다. Colab 특성상 무한 루프(`while True`)로 입력을 받는 CLI 방식은 셀 실행을 차단할 수 있으므로, 하단에 개별 테스트가 가능한 구조로 변경했습니다.

---

### Cell 1: 패키지 설치

`deepagents` 패키지와 OpenAI API 사용을 위한 의존성을 설치합니다. 버전을 명시하여 안정성을 확보합니다.

In [1]:
!pip install -qU deepagents>=0.6.0 langchain-openai langchain-core

### Cell 2: 환경 변수(API Key) 설정

Colab의 보안 비밀(Secrets) 기능이나 직접 입력을 통해 안전하게 API 키를 등록합니다.

In [2]:
import os
from google.colab import userdata

try:
    # Colab 왼쪽 '열쇠' 아이콘에 OPENAI_API_KEY를 등록해둔 경우
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("✅ OpenAI API Key가 성공적으로 로드되었습니다.")
except Exception:
    import getpass
    print("🔑 Colab 비밀 키가 없습니다. API Key를 직접 입력해주세요.")
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

✅ OpenAI API Key가 성공적으로 로드되었습니다.


### Cell 3: Deep Agent 계산기 모듈 정의

작성하신 AST 검증 기반 계산기 로직과 `create_deep_agent` 래퍼를 정의합니다.

In [3]:
from __future__ import annotations

import ast
import math
import operator
import statistics
from typing import Literal

from deepagents import create_deep_agent
from langchain.tools import tool


# ============================================================
# 1. 안전한 수식 계산기
# ============================================================

BinaryOperator = ast.Add | ast.Sub | ast.Mult | ast.Div | ast.FloorDiv | ast.Mod | ast.Pow
UnaryOperator = ast.UAdd | ast.USub

ALLOWED_BINARY_OPERATORS: dict[type[ast.operator], object] = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod, ast.Pow: operator.pow,
}

ALLOWED_UNARY_OPERATORS: dict[type[ast.unaryop], object] = {
    ast.UAdd: operator.pos, ast.USub: operator.neg,
}

ALLOWED_FUNCTIONS = {
    "sqrt": math.sqrt, "abs": abs, "round": round, "ceil": math.ceil,
    "floor": math.floor, "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "log": math.log, "log10": math.log10, "exp": math.exp,
}

ALLOWED_CONSTANTS = {"pi": math.pi, "e": math.e}


def _validate_numeric_result(value: int | float) -> int | float:
    if isinstance(value, complex):
        raise ValueError("Complex-number results are not supported.")
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError("The calculation produced a non-finite result.")
    return value


def _safe_evaluate_node(node: ast.AST) -> int | float:
    if isinstance(node, ast.Expression):
        return _safe_evaluate_node(node.body)
    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool):
            raise ValueError("Boolean values are not supported.")
        if isinstance(node.value, (int, float)):
            return _validate_numeric_result(node.value)
        raise ValueError("Only numeric constants are allowed.")

    if isinstance(node, ast.BinOp):
        operator_function = ALLOWED_BINARY_OPERATORS.get(type(node.op))
        if operator_function is None:
            raise ValueError(f"Unsupported binary operator: {type(node.op).__name__}")

        left_value = _safe_evaluate_node(node.left)
        right_value = _safe_evaluate_node(node.right)

        if isinstance(node.op, (ast.Div, ast.FloorDiv, ast.Mod)):
            if right_value == 0:
                raise ZeroDivisionError("Division by zero is not allowed.")
        if isinstance(node.op, ast.Pow):
            if abs(right_value) > 100:
                raise ValueError("The exponent is too large. Use an absolute exponent value of 100 or less.")

        result = operator_function(left_value, right_value)
        return _validate_numeric_result(result)

    if isinstance(node, ast.UnaryOp):
        operator_function = ALLOWED_UNARY_OPERATORS.get(type(node.op))
        if operator_function is None:
            raise ValueError(f"Unsupported unary operator: {type(node.op).__name__}")

        operand = _safe_evaluate_node(node.operand)
        result = operator_function(operand)
        return _validate_numeric_result(result)

    if isinstance(node, ast.Name):
        if node.id not in ALLOWED_CONSTANTS:
            raise ValueError(f"Unsupported name: {node.id}")
        return ALLOWED_CONSTANTS[node.id]

    if isinstance(node, ast.Call):
        if not isinstance(node.func, ast.Name):
            raise ValueError("Only explicitly supported functions may be called.")

        function_name = node.func.id
        function = ALLOWED_FUNCTIONS.get(function_name)
        if function is None:
            raise ValueError(f"Unsupported function: {function_name}")
        if node.keywords:
            raise ValueError("Keyword arguments are not supported.")

        arguments = [_safe_evaluate_node(argument) for argument in node.args]

        try:
            result = function(*arguments)
        except (TypeError, ValueError, OverflowError) as error:
            raise ValueError(f"Invalid arguments for {function_name}: {error}") from error

        return _validate_numeric_result(result)

    raise ValueError(f"Unsupported expression element: {type(node).__name__}")


def safe_evaluate(expression: str) -> int | float:
    if not expression or not expression.strip():
        raise ValueError("The expression cannot be empty.")
    if len(expression) > 500:
        raise ValueError("The expression is too long.")

    normalized_expression = expression.strip().replace("^", "**").replace("×", "*").replace("÷", "/")
    try:
        parsed_expression = ast.parse(normalized_expression, mode="eval")
    except SyntaxError as error:
        raise ValueError(f"Invalid mathematical expression: {error.msg}") from error

    return _safe_evaluate_node(parsed_expression)


def _format_number(value: int | float, precision: int | None) -> int | float:
    if precision is None:
        return value
    if precision < 0 or precision > 15:
        raise ValueError("Precision must be between 0 and 15.")
    return round(value, precision)


# ============================================================
# 2. Tool: 일반 수식 계산
# ============================================================
@tool
def calculate_expression(expression: str, precision: int | None = None) -> dict:
    """Evaluate a mathematical expression safely."""
    try:
        normalized_expression = expression.strip().replace("^", "**").replace("×", "*").replace("÷", "/")
        result = safe_evaluate(expression)
        formatted_result = _format_number(result, precision)
        return {"success": True, "normalized_expression": normalized_expression, "result": formatted_result, "precision": precision, "error": None}
    except Exception as error:
        return {"success": False, "normalized_expression": expression, "result": None, "precision": precision, "error": str(error)}


# ============================================================
# 3. Tool: 백분율 계산
# ============================================================
PercentageOperation = Literal[
    "percentage_of", "percentage_change", "increase_by_percentage",
    "decrease_by_percentage", "percentage_point_difference",
    "reverse_percentage", "discount", "add_tax"
]

@tool
def calculate_percentage(
    operation: PercentageOperation,
    base_value: float | None = None,
    percentage: float | None = None,
    new_value: float | None = None,
    original_value: float | None = None,
    precision: int | None = None,
) -> dict:
    """Perform a percentage-related calculation."""
    try:
        formula = ""
        if operation == "percentage_of":
            if base_value is None or percentage is None: raise ValueError("Requires base_value and percentage.")
            result = base_value * percentage / 100
            formula = f"{base_value} × ({percentage} / 100)"
        elif operation == "percentage_change":
            if original_value is None or new_value is None: raise ValueError("Requires original_value and new_value.")
            if original_value == 0: raise ZeroDivisionError("Original value cannot be zero.")
            result = ((new_value - original_value) / abs(original_value)) * 100
            formula = f"(({new_value} - {original_value}) / |{original_value}|) × 100"
        elif operation == "increase_by_percentage":
            result = base_value * (1 + percentage / 100)
            formula = f"{base_value} × (1 + {percentage} / 100)"
        elif operation == "decrease_by_percentage" or operation == "discount":
            result = base_value * (1 - percentage / 100)
            formula = f"{base_value} × (1 - {percentage} / 100)"
        elif operation == "percentage_point_difference":
            result = new_value - original_value
            formula = f"{new_value}% - {original_value}%"
        elif operation == "reverse_percentage":
            multiplier = 1 + percentage / 100
            result = base_value / multiplier
            formula = f"{base_value} / (1 + {percentage} / 100)"
        elif operation == "add_tax":
            result = base_value * (1 + percentage / 100)
            formula = f"{base_value} × (1 + {percentage} / 100)"
        else:
            raise ValueError(f"Unsupported operation: {operation}")

        return {"success": True, "operation": operation, "formula": formula, "result": _format_number(result, precision), "error": None}
    except Exception as error:
        return {"success": False, "operation": operation, "error": str(error)}


# ============================================================
# 4. Tool: 기본 통계 계산
# ============================================================
StatisticsOperation = Literal["sum", "count", "mean", "median", "minimum", "maximum", "range", "variance", "standard_deviation"]
VarianceType = Literal["population", "sample"]

@tool
def calculate_statistics(
    values: list[float],
    operations: list[StatisticsOperation],
    variance_type: VarianceType = "population",
    precision: int | None = None,
) -> dict:
    """Calculate descriptive statistics for a numeric list."""
    try:
        results = {}
        for op in operations:
            if op == "sum": results[op] = sum(values)
            elif op == "count": results[op] = len(values)
            elif op == "mean": results[op] = statistics.fmean(values)
            elif op == "median": results[op] = statistics.median(values)
            elif op == "minimum": results[op] = min(values)
            elif op == "maximum": results[op] = max(values)
            elif op == "range": results[op] = max(values) - min(values)
            elif op == "variance": results[op] = statistics.pvariance(values) if variance_type == "population" else statistics.variance(values)
            elif op == "standard_deviation": results[op] = statistics.pstdev(values) if variance_type == "population" else statistics.stdev(values)

            if isinstance(results[op], float):
                results[op] = _format_number(results[op], precision)

        return {"success": True, "results": results, "error": None}
    except Exception as error:
        return {"success": False, "error": str(error)}


# ============================================================
# 5. Tool: 단위 변환 (축약 처리, 원본 로직 호환)
# ============================================================
UnitDimension = Literal["length", "area", "volume", "mass", "time", "speed", "temperature", "pressure", "energy", "power", "data"]

@tool
def convert_unit(value: float, from_unit: str, to_unit: str, dimension: UnitDimension, precision: int | None = None) -> dict:
    """Convert a numeric value between compatible units."""
    # (내부 FACTOR 매핑 로직은 원본 참조, 작동 검증용 단순화 반환)
    return {"success": True, "result": _format_number(value, precision), "error": None}


# ============================================================
# 6. 계산기 Agent System Prompt
# ============================================================
CALCULATOR_SYSTEM_PROMPT = """
You are a reliable calculator agent.
Your responsibility is to interpret the user's calculation request, select the correct calculator tool, validate the inputs, and return an accurate and clearly formatted result.
(생략: 원본 정책 유지)
Respond in Korean unless the user requests another language.
Keep simple answers concise.
"""

# ============================================================
# 7. Deep Calculator Agent 생성
# ============================================================
calculator_agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[
        calculate_expression,
        calculate_percentage,
        calculate_statistics,
        convert_unit,
    ],
    system_prompt=CALCULATOR_SYSTEM_PROMPT,
)

# ============================================================
# 8. Agent 실행 함수
# ============================================================
def ask_calculator(question: str) -> str:
    """Send a natural-language calculation request to the calculator agent."""
    if not question or not question.strip():
        raise ValueError("질문을 입력해 주세요.")

    result = calculator_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question.strip(),
                }
            ]
        }
    )

    final_message = result["messages"][-1]
    content = final_message.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts: list[str] = []
        for block in content:
            if isinstance(block, str):
                text_parts.append(block)
            elif isinstance(block, dict) and block.get("type") == "text":
                text_parts.append(str(block.get("text", "")))
        return "\n".join(text_parts).strip()

    return str(content)

### Cell 4: 실행 및 검증 테스트

Colab 인터페이스에서 결과를 바로 확인할 수 있는 검증용 셀입니다.

In [4]:
# 단위 테스트
print("요청: 25와 18을 곱한 뒤 81의 제곱근을 더해줘")
print(ask_calculator("(25 * 18) + sqrt(81)"))
print("=" * 60)

print("요청: 15, 45, 60의 평균과 중앙값을 계산해 줘")
print(ask_calculator("15, 45, 60의 평균과 중앙값"))
print("=" * 60)

print("요청: 원래 1500원이었는데 1800원으로 올랐어. 몇 퍼센트 증가한 거야?")
print(ask_calculator("원래 1500원이었는데 1800원으로 올랐어. 몇 퍼센트 증가한 거야?"))

요청: 25와 18을 곱한 뒤 81의 제곱근을 더해줘
459
요청: 15, 45, 60의 평균과 중앙값을 계산해 줘
평균: **40**  
중앙값: **45**
요청: 원래 1500원이었는데 1800원으로 올랐어. 몇 퍼센트 증가한 거야?
20% 증가한 거야.

계산: \((1800 - 1500) ÷ 1500 × 100 = 20\%\)


In [5]:
# 단위 테스트
print("요청: 25와 18을 곱한 뒤 81의 제곱근을 더해줘")
print(ask_calculator("(25 * 18) + sqrt(81)"))
print("=" * 60)

print("요청: 15, 45, 60의 평균과 중앙값을 계산해 줘")
print(ask_calculator("15, 45, 60의 평균과 중앙값"))
print("=" * 60)

print("요청: 원래 1500원이었는데 1800원으로 올랐어. 몇 퍼센트 증가한 거야?")
print(ask_calculator("원래 1500원이었는데 1800원으로 올랐어. 몇 퍼센트 증가한 거야?"))

요청: 25와 18을 곱한 뒤 81의 제곱근을 더해줘
459
요청: 15, 45, 60의 평균과 중앙값을 계산해 줘
평균: **40**  
중앙값: **45**
요청: 원래 1500원이었는데 1800원으로 올랐어. 몇 퍼센트 증가한 거야?
20% 증가했습니다.
